# 03 · 15 March, small sample: refined grid around the best model

Notebook 02's best 15 March model (small sample) used `n_estimators=500, max_depth=10, min_samples_leaf=10,
max_features=0.7` (mean CV MAE 3.058 days). In that grid the number of trees barely mattered, so this notebook fixes
500 trees and searches more finely around the other three settings.

| | |
|---|---|
| Data | `model_features_mar15`, small sample (48 stations), development years 1964–2014, all 36 features |
| Target | anomaly from the 30-year station reference (same as notebook 02) |
| Grid | 27 candidates: `max_depth` 8/10/12 · `min_samples_leaf` 5/10/15 · `max_features` 0.6/0.7/0.8 · `n_estimators` 500 |
| Output | `results/rf/03_mar15_refined/` |
| Runtime | 135 fits, about 10 minutes on 2 CPUs |

Result: best `max_depth=8, min_samples_leaf=10, max_features=0.7` with mean CV MAE **3.0465** days (R² 0.881, ±3-day hit rate 58.5%).
This is the 15 March RF reported in the final report. Notebook 04 compares it with the notebook 02 result and makes the figures.

In [1]:
import time
from IPython.display import display
from sklearn.model_selection import ParameterGrid

from rf_common import *

OUTPUT_OVERRIDE = None   # None = results/rf/03_mar15_refined
N_JOBS = 2

OUT = output_dir('03_mar15_refined', OUTPUT_OVERRIDE)
PREFIX = 'mar15_small_refined'
DATASET = 'Mar15-small'

data = load_sample('mar15', 'small', development_only=True)
FEATURES = feature_columns(data)

if len(FEATURES) != 36:
    raise ValueError(f'Expected the 36 Mar15 features, found {len(FEATURES)}')
if not data['bloom_year'].between(1964, 2014).all():
    raise ValueError('Development rows must be 1964–2014')
if data.duplicated(['station_id', 'bloom_year']).any():
    raise ValueError('Duplicate station-year rows')
if not np.isfinite(data[FEATURES + [TARGET]].to_numpy(dtype=float)).all():
    raise ValueError('Missing or infinite values in features or target')

param_grid = {
    'n_estimators': [500],
    'max_depth': [8, 10, 12],
    'min_samples_leaf': [5, 10, 15],
    'max_features': [0.6, 0.7, 0.8],
}
candidates = list(ParameterGrid(param_grid))

save_json({
    'dataset': DATASET,
    'features': FEATURES,
    'param_grid': param_grid,
    'target': TARGET,
    'target_mode': 'anomaly: DOY minus 30-year station reference',
    'training_window': 'all training years of each fold',
    'station_reference_years': REFERENCE_YEARS,
    'minimum_station_records': MIN_HISTORY,
    'small_station_ids': sorted(data['station_id'].unique()),
    'random_state': RANDOM_STATE,
}, OUT / f'{PREFIX}_config.json')

print(f"{DATASET}: {data['station_id'].nunique()} stations, {len(data)} rows, {len(FEATURES)} features")
print(f'{len(candidates)} candidates × 5 folds = {len(candidates) * 5} fits')
print('Results go to:', OUT)

Mar15-small: 48 stations, 2444 rows, 36 features
27 candidates × 5 folds = 135 fits
Results go to: /home/claude/sakura_analytics/results/rf/03_mar15_refined


## Grid search

Saved after every candidate: all candidates (`_grid_search_all_candidates.csv`), every fold
(`_all_candidates_fold_results.csv`), and for the current best candidate its folds
(`_best_candidate_fold_results.csv`) and validation predictions (`_best_candidate_oof_predictions.csv`).

In [2]:
# Folds and station references are the same for every candidate.
fold_cache = []
for fold, train, valid in cv_folds(data):
    check_folds(train, valid, f'fold {fold}')
    fold_cache.append({'fold': fold, 'train': train, 'valid': valid,
                       'reference': make_reference(train)})

candidate_rows, all_fold_rows = [], []
best_mae, best_params = np.inf, None
started = time.perf_counter()

for candidate_id, params in enumerate(candidates):
    fold_rows, oof_parts = [], []

    for item in fold_cache:
        fold, train, valid = item['fold'], item['train'], item['valid']
        model, reference = fit_anomaly_rf(train, FEATURES, params,
                                          reference=item['reference'], n_jobs=N_JOBS)
        train_score = evaluate(train[TARGET], predict_doy(model, reference, train, FEATURES))
        valid_pred = predict_doy(model, reference, valid, FEATURES)
        valid_score = evaluate(valid[TARGET], valid_pred)

        fold_rows.append({
            'dataset': DATASET,
            'candidate_id': candidate_id,
            'fold': fold,
            **params,
            'train_start': int(train['bloom_year'].min()),
            'train_end': int(train['bloom_year'].max()),
            'reference_start': reference['start'],
            'reference_end': reference['end'],
            'valid_start': int(valid['bloom_year'].min()),
            'valid_end': int(valid['bloom_year'].max()),
            'train_rows': len(train),
            'valid_rows': len(valid),
            **{f'train_{m}': train_score[m] for m in METRICS},
            **{f'valid_{m}': valid_score[m] for m in METRICS},
        })

        part = valid[['station_id', 'location', 'region', 'bloom_year']].copy()
        part['fold'] = fold
        part['actual_doy'] = valid[TARGET].to_numpy()
        part['predicted_doy'] = valid_pred
        part['station_reference_doy'] = reference_values(reference, valid)
        part['error_days'] = part['predicted_doy'] - part['actual_doy']
        oof_parts.append(part)

    fold_table = pd.DataFrame(fold_rows)
    all_fold_rows.extend(fold_rows)

    # Candidates are ranked by the equal-weight mean of the five folds.
    candidate_rows.append({
        'dataset': DATASET,
        'candidate_id': candidate_id,
        'features': len(FEATURES),
        **params,
        **{f'train_{m}': fold_table[f'train_{m}'].mean() for m in METRICS},
        **{f'valid_{m}': fold_table[f'valid_{m}'].mean() for m in METRICS},
        'valid_MAE_std': fold_table['valid_MAE'].std(ddof=1),
    })
    grid_results = (
        pd.DataFrame(candidate_rows)
        .sort_values(['valid_MAE', 'candidate_id'], kind='stable')
        .reset_index(drop=True)
    )
    save_csv(grid_results, OUT / f'{PREFIX}_grid_search_all_candidates.csv')
    save_csv(pd.DataFrame(all_fold_rows), OUT / f'{PREFIX}_all_candidates_fold_results.csv')

    if candidate_rows[-1]['valid_MAE'] < best_mae:
        best_mae, best_params = candidate_rows[-1]['valid_MAE'], params
        save_csv(fold_table, OUT / f'{PREFIX}_best_candidate_fold_results.csv')
        save_csv(pd.concat(oof_parts, ignore_index=True),
                 OUT / f'{PREFIX}_best_candidate_oof_predictions.csv')

    if (candidate_id + 1) % 9 == 0:
        minutes = (time.perf_counter() - started) / 60
        print(f'{candidate_id + 1}/{len(candidates)} | best MAE {best_mae:.4f} | {minutes:.1f} min',
              flush=True)

print('Best parameters:', best_params)
display(grid_results.head(10).round(4))

9/27 | best MAE 3.0465 | 3.0 min


18/27 | best MAE 3.0465 | 6.3 min


27/27 | best MAE 3.0465 | 9.6 min


Best parameters: {'max_depth': 8, 'max_features': 0.7, 'min_samples_leaf': 10, 'n_estimators': 500}


,dataset,candidate_id,features,max_depth,max_features,min_samples_leaf,n_estimators,train_MAE,train_RMSE,train_R2,train_hit_rate_3d,valid_MAE,valid_RMSE,valid_R2,valid_hit_rate_3d,valid_MAE_std
0,Mar15-small,4,36,8,0.7,10,500,2.1754,2.8189,0.9330,0.7356,3.0465,3.6914,0.8808,0.5854,1.1252
1,Mar15-small,7,36,8,0.8,10,500,2.1649,2.8051,0.9336,0.7389,3.0494,3.6970,0.8804,0.5792,1.1152
2,Mar15-small,5,36,8,0.7,15,500,2.3109,2.9952,0.9243,0.7081,3.0547,3.7017,0.8803,0.5833,1.1426
3,Mar15-small,8,36,8,0.8,15,500,2.3018,2.9851,0.9248,0.7084,3.0562,3.7042,0.8802,0.5833,1.1279
4,Mar15-small,13,36,10,0.7,10,500,2.0148,2.6411,0.9411,0.7746,3.0578,3.7026,0.8805,0.5875,1.0968
5,Mar15-small,1,36,8,0.6,10,500,2.1814,2.8264,0.9326,0.7328,3.0606,3.7006,0.8805,0.5792,1.1476
6,Mar15-small,10,36,10,0.6,10,500,2.0280,2.6563,0.9405,0.7718,3.0607,3.6976,0.8806,0.5792,1.1222
7,Mar15-small,16,36,10,0.8,10,500,2.0057,2.6286,0.9417,0.7757,3.0609,3.7035,0.8800,0.5729,1.0949
8,Mar15-small,26,36,12,0.8,15,500,2.1926,2.8703,0.9305,0.7323,3.0634,3.7064,0.8802,0.5833,1.1054
9,Mar15-small,25,36,12,0.8,10,500,1.9526,2.5744,0.9441,0.7897,3.0634,3.7057,0.8800,0.5729,1.0704
